In [ ]:
import numpy as np
import cv2

'''=====圖片路徑更改====='''
image1_path = '../Data/quiz4/left.png'
image2_path = '../Data/quiz4/right.png'

image_left = cv2.imread(image1_path)
image_right = cv2.imread(image2_path)

#SIFT 用灰階會比較好?
gray_left = cv2.cvtColor(image_left, cv2.COLOR_BGR2GRAY)
gray_right = cv2.cvtColor(image_right, cv2.COLOR_BGR2GRAY)

# 建立 SIFT 偵測器
sift = cv2.SIFT_create()

# 偵測特徵點並計算描述子
keypoints_left, descriptors_left = sift.detectAndCompute(gray_left, None)
keypoints_right, descriptors_right = sift.detectAndCompute(gray_right, None)

print(f"左圖特徵點數量：{len(keypoints_left)}")
print(f"右圖特徵點數量：{len(keypoints_right)}")

# 繪製特徵點：圓圈表示尺度，線段表示方向
result_left = cv2.drawKeypoints(
    image_left, keypoints_left, None,
    flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS
)

result_right = cv2.drawKeypoints(
    image_right, keypoints_right, None,
    flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS
)

#cv2.imwrite("left_sift.png", result_left)
#cv2.imwrite("right_sift.png", result_right)

左圖特徵點數量：698
右圖特徵點數量：693


In [60]:
# 確認有足夠描述子可進行匹配
if (
    descriptors_left is None
    or descriptors_right is None
    or len(descriptors_right) < 2
):
    raise ValueError("特徵點不足，無法進行雙鄰居匹配")

# SIFT 描述子使用 L2 距離比較
matcher = cv2.BFMatcher(cv2.NORM_L2)

# 對每個左圖特徵，找出右圖中最相似的兩個特徵
matches = matcher.knnMatch(
    descriptors_left,
    descriptors_right,
    k=2
)

# Lowe's ratio test：保留明顯優於第二名的配對
good_matches = []

for pair in matches:
    if len(pair) < 2:
        continue

    m, n = pair  # m 是最近鄰，n 是次近鄰

    if m.distance < 0.75 * n.distance:
        good_matches.append(m)

print(f"初始匹配組數：{len(matches)}")
print(f"篩選後配對數：{len(good_matches)}")

# 繪製篩選後的配對
matched_image = cv2.drawMatches(
    image_left, keypoints_left,
    image_right, keypoints_right,
    good_matches, None,
    flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS
)

#cv2.imwrite("sift_matches.png", matched_image)

初始匹配組數：698
篩選後配對數：372


In [61]:
# 單應性矩陣至少需要 4 組配對
if len(good_matches) < 4:
    raise ValueError("有效配對不足 4 組，無法估計單應性矩陣")

# 取得配對點座標
# queryIdx 對應左圖，trainIdx 對應右圖
points_left = np.float32([
    keypoints_left[m.queryIdx].pt for m in good_matches
]).reshape(-1, 1, 2)

points_right = np.float32([
    keypoints_right[m.trainIdx].pt for m in good_matches
]).reshape(-1, 1, 2)

# 使用 RANSAC 估計 H：將右圖座標映射到左圖
H, mask = cv2.findHomography(
    points_right,
    points_left,
    method=cv2.RANSAC,
    ransacReprojThreshold=3.0
)

if H is None or mask is None:
    raise ValueError("單應性矩陣估計失敗，請檢查影像與配對品質")

# mask 中 1 表示內點，0 表示被排除的外點
inlier_mask = mask.ravel().astype(bool)

inlier_matches = [
    m for m, is_inlier in zip(good_matches, inlier_mask)
    if is_inlier
]

print(f"RANSAC 前配對數：{len(good_matches)}")
print(f"RANSAC 後內點數：{len(inlier_matches)}")

# 只繪製 RANSAC 保留的配對
ransac_image = cv2.drawMatches(
    image_left, keypoints_left,
    image_right, keypoints_right,
    inlier_matches, None,
    flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS
)

#cv2.imwrite("ransac_matches.png", ransac_image)

RANSAC 前配對數：372
RANSAC 後內點數：370


In [ ]:
# 取得兩張影像的尺寸
h_left, w_left = image_left.shape[:2]
h_right, w_right = image_right.shape[:2]

# 兩張影像的四個角落座標
corners_left = np.float32([
    [0, 0], [w_left, 0],
    [w_left, h_left], [0, h_left]
]).reshape(-1, 1, 2)

corners_right = np.float32([
    [0, 0], [w_right, 0],
    [w_right, h_right], [0, h_right]
]).reshape(-1, 1, 2)

# 將右圖四角映射到左圖座標系
warped_corners_right = cv2.perspectiveTransform(corners_right, H)

# 計算可容納兩張影像的畫布範圍
all_corners = np.concatenate(
    [corners_left, warped_corners_right], axis=0
)

x_min, y_min = np.floor(
    all_corners.min(axis=(0, 1))
).astype(int)

x_max, y_max = np.ceil(
    all_corners.max(axis=(0, 1))
).astype(int)

canvas_width = int(x_max - x_min)
canvas_height = int(y_max - y_min)

# 平移矩陣：將可能的負座標移到畫布內
T = np.array([
    [1, 0, -x_min],
    [0, 1, -y_min],
    [0, 0, 1]
], dtype=np.float64)

# 右圖先透視變換 H，再平移 T
warped_right = cv2.warpPerspective(
    image_right,
    T @ H,
    (canvas_width, canvas_height)
)

# 左圖只需平移至同一畫布
warped_left = cv2.warpPerspective(
    image_left,
    T,
    (canvas_width, canvas_height)
)

# 初步拼接：將左圖放到對齊後的右圖上
panorama = warped_right.copy()

offset_x = int(-x_min)
offset_y = int(-y_min)

panorama[
    offset_y:offset_y + h_left,
    offset_x:offset_x + w_left
] = image_left



True

In [ ]:
'''=====儲存合併影像結果路徑====='''
cv2.imwrite("panorama.png", panorama)